# Notebook 2 — Feature Engineering & Preprocessing

**What this notebook does:** Engineers new predictive features, encodes categorical
variables, splits the data, balances the training set with SMOTE, and scales
features ready for model training.

**Input:** `data/processed/cleaned_churn.csv`

**Output:**
- `data/processed/X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`
- `models/scaler.pkl`, `models/feature_names.pkl`

**Estimated run time:** ~15 seconds

In [1]:
# Imports and shared project utilities
import os
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from utils import save_dataframe, save_joblib_object, load_csv_safely, project_path

In [2]:
# Load the cleaned dataset produced by notebook 1
cleaned_path = project_path("data", "processed", "cleaned_churn.csv")
df = load_csv_safely(cleaned_path, "01_data_cleaning_eda.ipynb")
print("Loaded shape:", df.shape)
df.head()

Loaded shape: (7032, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## 1. Feature engineering

In [3]:
# tenure_group: bin tenure (in months) into 6 yearly bands
tenure_bins = [0, 12, 24, 36, 48, 60, 72]
tenure_labels = ["0-12 months", "13-24 months", "25-36 months",
                 "37-48 months", "49-60 months", "61-72 months"]
df["tenure_group"] = pd.cut(df["tenure"], bins=tenure_bins, labels=tenure_labels, include_lowest=True)

# avg_monthly_spend: total spend normalised by tenure (+1 to avoid division by zero)
df["avg_monthly_spend"] = df["TotalCharges"] / (df["tenure"] + 1)

# is_long_term_contract: 1 if the customer is on an annual or biennial contract
df["is_long_term_contract"] = df["Contract"].isin(["One year", "Two year"]).astype(int)

# high_value_customer: 1 if MonthlyCharges is above the dataset median
df["high_value_customer"] = (df["MonthlyCharges"] > df["MonthlyCharges"].median()).astype(int)

# num_services: count of subscribed add-on/core services (Yes = 1, else 0)
service_columns = ["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
                   "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
df["num_services"] = df[service_columns].apply(lambda col: (col == "Yes").astype(int)).sum(axis=1)

print("New columns added: tenure_group, avg_monthly_spend, is_long_term_contract, high_value_customer, num_services")
df[["tenure_group", "avg_monthly_spend", "is_long_term_contract", "high_value_customer", "num_services"]].head()

New columns added: tenure_group, avg_monthly_spend, is_long_term_contract, high_value_customer, num_services


,tenure_group,avg_monthly_spend,is_long_term_contract,high_value_customer,num_services
0,0-12 months,14.925000,0,0,1
1,25-36 months,53.985714,1,0,3
2,0-12 months,36.050000,0,0,3
3,37-48 months,40.016304,1,0,3
4,0-12 months,50.550000,0,1,1


## 2. Encoding

In [4]:
# Binary Yes/No columns -> 1/0 (any 'No xxx service' variant is treated as No -> 0)
binary_columns = ["Partner", "Dependents", "PhoneService", "PaperlessBilling",
                   "OnlineSecurity", "OnlineBackup", "DeviceProtection",
                   "TechSupport", "StreamingTV", "StreamingMovies", "SeniorCitizen"]

for col in binary_columns:
    df[col] = (df[col] == "Yes").astype(int)

print("Binary-encoded columns:", binary_columns)

Binary-encoded columns: ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'SeniorCitizen']


In [5]:
# One-hot encode multi-category columns, dropping the first level to avoid the dummy trap
onehot_columns = ["gender", "MultipleLines", "InternetService", "Contract", "PaymentMethod"]
df = pd.get_dummies(df, columns=onehot_columns, drop_first=True)

print(f"Shape after one-hot encoding: {df.shape}")

Shape after one-hot encoding: (7032, 31)


In [6]:
# Drop identifier and the binning-only tenure_group column (raw tenure is kept as a numeric feature)
df_model = df.drop(columns=["customerID", "tenure_group"])

# One-hot dummy columns are created as bool; cast everything to numeric for modelling
df_model = df_model.astype({col: int for col in df_model.select_dtypes(include="bool").columns})

X = df_model.drop(columns=["Churn"])
y = df_model["Churn"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (7032, 28)
Target shape: (7032,)


## 3. Train/test split, SMOTE balancing, and scaling

In [7]:
# 80/20 stratified split so both sets preserve the ~26.6% churn rate
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Class distribution before SMOTE (training set):")
print(y_train.value_counts())
print(f"Churn rate in training set: {y_train.mean() * 100:.1f}%")

Class distribution before SMOTE (training set):
Churn
0    4130
1    1495
Name: count, dtype: int64
Churn rate in training set: 26.6%


In [8]:
# Apply SMOTE to the training set only — the test set must reflect real-world class balance
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Class distribution after SMOTE (training set):")
print(y_train_resampled.value_counts())
print(f"Churn rate in resampled training set: {y_train_resampled.mean() * 100:.1f}%")

Class distribution after SMOTE (training set):
Churn
0    4130
1    4130
Name: count, dtype: int64
Churn rate in resampled training set: 50.0%


In [9]:
# Scale features: fit the scaler on the (SMOTE-balanced) training set only, then
# apply the same transform to the test set
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_resampled), columns=X_train_resampled.columns
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns
)

scaler_path = project_path("models", "scaler.pkl")
save_joblib_object(scaler, scaler_path)

feature_names = list(X.columns)
feature_names_path = project_path("models", "feature_names.pkl")
save_joblib_object(feature_names, feature_names_path)

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/models/scaler.pkl (2.2KB)
✅ Saved: /Users/prasadlola/Documents/customer_churn_project/models/feature_names.pkl (568.0B)


## 4. Save processed datasets

In [10]:
# Save the train/test splits for use in notebook 3 (model training)
save_dataframe(X_train_scaled, project_path("data", "processed", "X_train.csv"))
save_dataframe(X_test_scaled, project_path("data", "processed", "X_test.csv"))
save_dataframe(y_train_resampled.to_frame(name="Churn"), project_path("data", "processed", "y_train.csv"))
save_dataframe(y_test.to_frame(name="Churn"), project_path("data", "processed", "y_test.csv"))

print("\nFinal shapes:")
print("X_train:", X_train_scaled.shape)
print("X_test :", X_test_scaled.shape)
print("y_train:", y_train_resampled.shape)
print("y_test :", y_test.shape)

✅ Saved: /Users/prasadlola/Documents/customer_churn_project/data/processed/X_train.csv (4.3MB) — 8260 rows, 28 columns


✅ Saved: /Users/prasadlola/Documents/customer_churn_project/data/processed/X_test.csv (756.5KB) — 1407 rows, 28 columns
✅ Saved: /Users/prasadlola/Documents/customer_churn_project/data/processed/y_train.csv (16.1KB) — 8260 rows, 1 columns
✅ Saved: /Users/prasadlola/Documents/customer_churn_project/data/processed/y_test.csv (2.8KB) — 1407 rows, 1 columns

Final shapes:
X_train: (8260, 28)
X_test : (1407, 28)
y_train: (8260,)
y_test : (1407,)
